# Hybrid Search and Reranking Tutorial

## Overview

This notebook demonstrates three different search approaches for retrieving relevant documents from a vector database:

1. **Pure Vector Search** - Uses only semantic similarity (embeddings)
2. **Hybrid Search** - Combines vector search with keyword/BM25 search
3. **Hybrid Search + Reranking** - Adds a reranking step for improved accuracy

---

## Understanding the Three Search Methods

### 1. Pure Vector Search
- **How it works**: Converts the query into an embedding vector and finds documents with similar vectors
- **Pros**: Captures semantic meaning (e.g., "car" matches "automobile")
- **Cons**: May miss exact keyword matches

### 2. Hybrid Search (Vector + BM25)
- **How it works**: Combines vector similarity with traditional keyword matching (BM25 algorithm)
- **Pros**: Best of both worlds - semantic understanding + exact matches
- **Cons**: Slightly slower than pure vector search

### 3. Hybrid Search + Reranking
- **How it works**: After hybrid search, a reranker model scores and reorders results
- **Pros**: Highest quality results (~15% improvement)
- **Cons**: Additional latency (1-1.5s)

---

## When to Use Each Method

| Method | Best For | Trade-off |
|--------|----------|----------|
| Pure Vector | High-throughput search (>5 QPS) | Speed over precision |
| Hybrid | Balanced applications | Good quality + reasonable speed |
| Hybrid + Reranking | RAG systems, quality-critical apps | Quality over speed |

## Step 1: Install Required Dependencies

We need the following packages:
- **langchain**: Framework for building LLM applications
- **langchain-huggingface**: HuggingFace integration for embeddings
- **sentence-transformers**: For generating text embeddings
- **databricks-langchain**: Databricks-specific integrations

In [ ]:
# ============================================================================
# INSTALL DEPENDENCIES
# ============================================================================
# These packages are required for embedding generation and vector search

%pip install langchain langchain-huggingface langchain-community sentence-transformers
%pip install databricks-langchain

# Restart Python kernel to load new packages (Databricks-specific)
dbutils.library.restartPython()

## Step 2: Import Libraries and Initialize Components

We'll import the necessary libraries and set up:
- **VectorSearchClient**: Databricks client for vector database operations
- **DatabricksReranker**: Built-in reranker for improving search results
- **HuggingFaceEmbeddings**: Embedding model for converting text to vectors
- **MLflow**: For experiment tracking and evaluation

In [ ]:
# ============================================================================
# IMPORT LIBRARIES
# ============================================================================

# MLflow for experiment tracking and evaluation metrics
import mlflow
from mlflow.entities import Document
from mlflow.genai.scorers import RetrievalRelevance

# Databricks vector search components
from databricks.vector_search.client import VectorSearchClient
from databricks.vector_search.reranker import DatabricksReranker

# HuggingFace embeddings for text-to-vector conversion
from langchain_huggingface import HuggingFaceEmbeddings

# Standard libraries
from typing import List
import pandas as pd

print("Libraries imported successfully!")

## Step 3: Initialize Vector Search Client and Embedding Model

### About the Embedding Model: BAAI/bge-large-en-v1.5

We use the **BGE (BAAI General Embedding)** model, which is:
- One of the top-performing open-source embedding models
- Produces 1024-dimensional vectors
- Optimized for retrieval tasks
- Supports L2 normalization for cosine similarity

In [ ]:
# ============================================================================
# INITIALIZE CLIENTS AND MODELS
# ============================================================================

# Initialize the Databricks Vector Search client
# This client connects to your Databricks workspace's vector search service
client = VectorSearchClient()

# Get current username for MLflow experiment path
# This ensures each user has their own experiment namespace
username = spark.sql("SELECT current_user()").first()[0]
experiment_path = f"/Users/{username}/hybrid_search_reranking"

# Set up MLflow experiment for tracking search quality metrics
mlflow.set_experiment(experiment_path)
print(f"MLflow experiment set to: {experiment_path}")

# ============================================================================
# INITIALIZE EMBEDDING MODEL
# ============================================================================
# BGE (BAAI General Embedding) is a state-of-the-art embedding model
# 
# Key parameters:
# - model_name: The HuggingFace model identifier
# - device: 'cpu' or 'cuda' for GPU acceleration
# - normalize_embeddings: True enables cosine similarity via dot product

bge_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",  # 1024-dimensional embeddings
    model_kwargs={'device': 'cpu'},        # Use CPU (change to 'cuda' for GPU)
    encode_kwargs={'normalize_embeddings': True}  # L2 normalize for cosine sim
)

print("Embedding model initialized successfully!")
print(f"Model: BAAI/bge-large-en-v1.5")
print(f"Embedding dimension: 1024")

## Step 4: Define Test Queries

We'll use a set of AI/ML-related queries to test and compare the different search methods.

In [ ]:
# ============================================================================
# TEST QUERIES
# ============================================================================
# A diverse set of queries covering different AI/ML topics
# These will help us evaluate how each search method performs

test_queries = [
    "machine learning algorithms",       # General ML query
    "embeddings vector databases",       # Vector DB specific
    "neural networks deep learning",     # Deep learning focus
    "transformers attention mechanism",  # Transformer architecture
    "NLP natural language processing",   # NLP domain
]

print("Test queries defined:")
for i, query in enumerate(test_queries, 1):
    print(f"  {i}. {query}")

---

# Search Method 1: Pure Vector Search

## How Pure Vector Search Works

```
Query: "machine learning algorithms"
         |
         v
   [Embedding Model]
         |
         v
   Query Vector: [0.12, -0.34, 0.56, ...] (1024 dimensions)
         |
         v
   [Vector Database] -- Compare with document vectors
         |
         v
   Top-K Most Similar Documents
```

### Key Characteristics:
- **Semantic Understanding**: Matches concepts, not just words
- **Fast**: Single vector similarity computation
- **Limitation**: May miss documents with exact keyword matches but different phrasing

In [ ]:
# ============================================================================
# METHOD 1: PURE VECTOR SEARCH
# ============================================================================
# This method uses ONLY semantic similarity to find relevant documents.
# It converts the query to a vector and finds documents with similar vectors.

def pure_vector_search(query: str, num_results: int = 5) -> List[dict]:
    """
    Perform pure vector similarity search.
    
    This method:
    1. Converts the query text into an embedding vector
    2. Searches the vector index for similar document vectors
    3. Returns the top-K most similar documents
    
    Args:
        query (str): The search query text
        num_results (int): Number of results to return (default: 5)
    
    Returns:
        List[dict]: List of documents with id, text, score, and method
    
    Example:
        >>> results = pure_vector_search("machine learning")
        >>> print(results[0]['text'])
    """
    
    # Step 1: Convert query text to embedding vector
    # The embedding model transforms text into a dense vector representation
    query_vector = bge_model.embed_query(query)
    
    # Step 2: Get the vector search index
    # The index contains pre-computed embeddings of all documents
    index = client.get_index(index_name="users.anirvan_sen.bge_embedding_vs_index")
    
    # Step 3: Perform similarity search
    # This computes cosine similarity between query vector and all document vectors
    results = index.similarity_search(
        query_vector=query_vector,  # The query embedding
        columns=["id", "text"],     # Columns to return from matching documents
        num_results=num_results     # Number of top results to return
    )
    
    # Step 4: Parse and format the results
    docs = []
    for row in results.get('result', {}).get('data_array', []):
        docs.append({
            'id': str(row[0]),                           # Document ID
            'text': row[1],                              # Document text content
            'score': row[-1] if len(row) > 2 else None,  # Similarity score
            'method': 'vector_only'                      # Search method used
        })
    
    return docs


# ============================================================================
# TEST PURE VECTOR SEARCH
# ============================================================================
print("=" * 80)
print("PURE VECTOR SEARCH RESULTS")
print("=" * 80)

# Run a test query
test_query =-
print(f"\nQuery: '{test_query}'\n")

vector_results = pure_vector_search(test_query)

# Display results with ranking and scores
for i, doc in enumerate(vector_results, 1):
    print(f"{i}. Score: {doc['score']:.4f} | {doc['text'][:60]}...")

print("\n" + "-" * 80)
print("Note: Pure vector search finds semantically similar documents,")
print("but may miss exact keyword matches.")

# Search Method 2: Hybrid Search (Vector + BM25)

## How Hybrid Search Works

```
Query: "machine learning algorithms"
         |
         +-------------------+
         |                   |
         v                   v
   [Vector Search]     [BM25 Keyword Search]
         |                   |
         v                   v
   Semantic Matches    Keyword Matches
         |                   |
         +-------+   +-------+
                 |   |
                 v   v
            [Score Fusion]
                   |
                   v
           Combined Results
```

### What is BM25?
BM25 (Best Matching 25) is a ranking function that:
- Considers term frequency (TF): How often the query terms appear in a document
- Considers inverse document frequency (IDF): How rare the terms are across all documents
- Penalizes very long documents

### Why Combine Vector + BM25?
| Scenario | Vector Search | BM25 | Hybrid |
|----------|--------------|------|--------|
| "What is ML?" matching "machine learning explained" | ✓ Great | ✗ Poor | ✓ Great |
| Exact term "BERT-base" | ✗ Poor | ✓ Great | ✓ Great |

In [ ]:
# ============================================================================
# METHOD 2: HYBRID SEARCH (Vector + BM25 Keyword)
# ============================================================================
# This method combines semantic vector search with traditional keyword matching.
# It provides better coverage than pure vector search alone.

def hybrid_search(query: str, num_results: int = 5) -> List[dict]:
    """
    Perform hybrid search combining vector similarity and keyword matching.
    
    This method:
    1. Converts the query to an embedding vector (for semantic search)
    2. Also uses the raw query text (for keyword/BM25 search)
    3. Fuses both scores to produce final rankings
    
    The key difference from pure vector search is the addition of:
    - query_text parameter: Used for BM25 keyword matching
    - query_type="hybrid": Tells the index to use both methods
    
    Args:
        query (str): The search query text
        num_results (int): Number of results to return (default: 5)
    
    Returns:
        List[dict]: List of documents with id, text, score, and method
    """
    
    # Step 1: Convert query to embedding (same as pure vector search)
    query_vector = bge_model.embed_query(query)
    
    # Step 2: Get the vector search index
    index = client.get_index(index_name="users.anirvan_sen.bge_embedding_vs_index")
    
    # Step 3: Perform HYBRID similarity search
    # KEY DIFFERENCES from pure vector search:
    # - query_text: The raw text for BM25 keyword matching
    # - query_type="hybrid": Activates both vector + keyword search
    results = index.similarity_search(
        query_vector=-,  # For vector similarity
        query_text=query,           # For BM25 keyword matching (NEW!)
        columns=["id", "text"],
        num_results=num_results,
        query_type="hybrid"         # Enable hybrid mode (NEW!)
    )
    
    # Step 4: Parse and format results
    docs = []
    for row in results.get('result', {}).get('data_array', []):
        docs.append({
            'id': str(row[0]),
            'text': row[1],
            'score': row[-1] if len(row) > 2 else None,
            'method': 'hybrid'  # Mark as hybrid search
        })
    
    return docs


# ============================================================================
# TEST HYBRID SEARCH
# ============================================================================
print("=" * 80)
print("HYBRID SEARCH RESULTS (Vector + BM25)")
print("=" * 80)

test_query = "machine learning algorithms"
print(f"\nQuery: '{test_query}'\n")

hybrid_results = hybrid_search(test_query)

# Display results
for i, doc in enumerate(hybrid_results, 1):
    print(f"{i}. Score: {doc['score']:.4f} | {doc['text'][:60]}...")

print("\n" + "-" * 80)
print("Note: Hybrid search combines semantic meaning with exact keyword matches.")
print("This often produces more comprehensive results than pure vector search.")

---

# Search Method 3: Hybrid Search + Reranking

## How Reranking Works

```
Query: "machine learning algorithms"
         |
         v
   [Hybrid Search] -- Returns top-K candidates
         |
         v
   Candidate Documents (e.g., 20-50 docs)
         |
         v
   [Reranker Model] -- Cross-encoder scoring
         |
         v
   Reordered Results (highest quality on top)
```

### What is a Reranker?

A reranker is a **cross-encoder model** that:
1. Takes the query AND each document as input together
2. Computes a relevance score for each (query, document) pair
3. Reorders results based on these more accurate scores

### Bi-Encoder vs Cross-Encoder

| Aspect | Bi-Encoder (Embeddings) | Cross-Encoder (Reranker) |
|--------|------------------------|-------------------------|
| Speed | Fast (pre-computed) | Slow (computed at query time) |
| Accuracy | Good | Excellent |
| Scalability | Billions of docs | Hundreds of candidates |
| Use Case | First-stage retrieval | Second-stage reranking |

### Why Use Reranking?
- **~15% improvement** in retrieval quality
- Catches nuances that embeddings miss
- Worth the latency for quality-critical applications (RAG, chatbots)

In [ ]:
# ============================================================================
# METHOD 3: HYBRID SEARCH + RERANKING (Best Quality)
# ============================================================================
# This method adds a reranking step after hybrid search.
# The reranker uses a cross-encoder model to score each (query, doc) pair.

def hybrid_search_with_reranking(query: str, num_results: int = 5) -> List[dict]:
    """
    Perform hybrid search with built-in Databricks reranker.
    
    This method:
    1. Performs hybrid search (vector + BM25)
    2. Retrieves more candidates than needed
    3. Uses a reranker model to rescore and reorder results
    
    The reranker (cross-encoder) provides more accurate relevance scores
    by processing the query and document together, allowing it to capture
    fine-grained semantic relationships.
    
    Args:
        query (str): The search query text
        num_results (int): Number of final results to return (default: 5)
    
    Returns:
        List[dict]: List of documents with id, text, score, and method
    
    Note:
        Reranking adds 1-1.5 seconds of latency but improves quality by ~15%
    """
    
    # Step 1: Convert query to embedding
    query_vector = bge_model.embed_query(query)
    
    # Step 2: Get the vector search index
    index = client.get_index(index_name="users.anirvan_sen.bge_embedding_vs_index")
    
    # Step 3: Perform hybrid search WITH RERANKING
    # KEY ADDITION: The 'reranker' parameter
    # - DatabricksReranker: Uses Databricks' built-in reranking model
    # - columns_to_rerank: Which columns to use for reranking (typically text)
    results = index.similarity_search(
        query_vector=query_vector,
        query_text=query,
        columns=["id", "text"],
        num_results=num_results,
        query_type="hybrid",
        # NEW: Add reranker for quality improvement
        reranker=DatabricksReranker(
            columns_to_rerank=["text"]  # Rerank based on text content
        )
    )
    
    # Step 4: Parse and format results
    docs = []
    for row in results.get('result', {}).get('data_array', []):
        docs.append({
            'id': str(row[0]),
            'text': row[1],
            'score': row[-1] if len(row) > 2 else None,
            'method': 'hybrid_reranked'  # Mark as hybrid + reranked
        })
    
    return docs


# ============================================================================
# TEST HYBRID SEARCH + RERANKING
# ============================================================================
print("=" * 80)
print("HYBRID SEARCH + RERANKING RESULTS (Best Quality)")
print("=" * 80)

test_query = "machine learning algorithms"
print(f"\nQuery: '{test_query}'\n")

reranked_results = hybrid_search_with_reranking(test_query)

# Display results
for i, doc in enumerate(reranked_results, 1):
    print(f"{i}. Score: {doc['score']:.4f} | {doc['text'][:60]}...")

print("\n" + "-" * 80)
print("Note: Reranking provides the highest quality results.")
print("The cross-encoder model captures nuanced query-document relationships.")

---

# Comparison: All Three Methods

Now let's compare the three search methods side by side to understand their trade-offs.

In [ ]:
# ============================================================================
# COMPARISON TABLE: All Three Methods
# ============================================================================
# This comparison helps you choose the right method for your use case

print("=" * 80)
print("COMPARISON: All Three Search Methods")
print("=" * 80 + "\n")

# Create a comparison DataFrame for easy visualization
comparison_data = {
    'Method': [
        'Pure Vector Search',
        'Hybrid Search',
        'Hybrid + Reranking'
    ],
    'Uses Vector': ['✓', '✓', '✓'],
    'Uses Keywords/BM25': ['✗', '✓', '✓'],
    'Uses Reranker': ['✗', '✗', '✓'],
    'Quality': ['Good', 'Better', 'Best ⭐'],
    'Speed': ['Fast', 'Medium', 'Slower (1-1.5s)'],
}

comparison_df = pd.DataFrame(comparison_data)

# Display the comparison table
display(comparison_df)

# Additional context
print("\n" + "=" * 80)
print("DETAILED BREAKDOWN")
print("=" * 80)

print("""
┌─────────────────────────────────────────────────────────────────────────────┐
│                          SEARCH METHOD COMPARISON                           │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  1. PURE VECTOR SEARCH                                                      │
│     ├─ What it does: Semantic similarity only                               │
│     ├─ Best for: Real-time search bars, high throughput (>5 QPS)           │
│     ├─ Pros: Fastest, understands synonyms and concepts                     │
│     └─ Cons: May miss exact keyword matches                                 │
│                                                                             │
│  2. HYBRID SEARCH (Vector + BM25)                                           │
│     ├─ What it does: Combines semantic + keyword matching                   │
│     ├─ Best for: Balanced quality & speed, mixed query types               │
│     ├─ Pros: Catches both semantic meaning AND exact terms                  │
│     └─ Cons: Slight latency overhead                                        │
│                                                                             │
│  3. HYBRID SEARCH + RERANKING ⭐ (Recommended for RAG)                       │
│     ├─ What it does: Hybrid search + cross-encoder reranking               │
│     ├─ Best for: RAG agents, quality-critical applications                 │
│     ├─ Pros: ~15% quality improvement, most accurate results               │
│     └─ Cons: 1-1.5s latency (acceptable for most RAG use cases)            │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
""")

---

# Recommendations

## Choosing the Right Search Method

Use this decision tree to select the best method for your use case:

```
Is latency critical (<100ms required)?
├─ YES → Use Pure Vector Search
└─ NO
    ├─ Is this for a RAG/chatbot application?
    │   ├─ YES → Use Hybrid + Reranking ⭐
    │   └─ NO → Use Hybrid Search
    └─ Do you need exact keyword matching?
        ├─ YES → Use Hybrid Search (minimum)
        └─ NO → Pure Vector Search is fine
```

In [ ]:
# ============================================================================
# RECOMMENDATIONS SUMMARY
# ============================================================================

print("=" * 80)
print("RECOMMENDATIONS")
print("=" * 80 + "\n")

print("""
┌─────────────────────────────────────────────────────────────────────────────┐
│                           WHEN TO USE EACH METHOD                           │
├─────────────────────────────────────────────────────────────────────────────┤

  1️⃣  PURE VECTOR SEARCH
      ├─ Best for: Real-time search bars, high throughput (>5 QPS)
      ├─ Pros: Fastest, semantically smart
      └─ Cons: Misses exact keyword matches

  2️⃣  HYBRID SEARCH (Vector + BM25)
      ├─ Best for: Balanced quality & speed, mixed queries
      ├─ Pros: Catches keywords + semantic meaning
      └─ Cons: Slight latency overhead

  3️⃣  HYBRID SEARCH + RERANKING ⭐ (RECOMMENDED FOR RAG)
      ├─ Best for: RAG agents, quality-critical applications
      ├─ Pros: ~15% quality improvement, best results
      └─ Cons: 1-1.5s latency (acceptable for RAG)

└─────────────────────────────────────────────────────────────────────────────┘

📌 KEY TAKEAWAYS:

   • For most RAG applications, use Hybrid + Reranking
   • The 1-1.5s latency is usually acceptable for chatbot-style interactions
   • Quality improvements from reranking typically outweigh the speed cost
   • Pure vector search is best when you need sub-100ms responses
""")

print("\n✅ Tutorial Complete!")
print("\nNext steps:")
print("  1. Try each method with your own data")
print("  2. Measure quality using MLflow evaluations")
print("  3. Choose the method that balances quality and latency for your use case")

---

# Summary

## What We Learned

1. **Pure Vector Search** is fast and captures semantic meaning but may miss exact matches

2. **Hybrid Search** combines the best of vector and keyword search for better coverage

3. **Hybrid + Reranking** provides the highest quality results at the cost of some latency

## Key Code Differences

```python
# Pure Vector Search
results = index.similarity_search(
    query_vector=query_vector,
    num_results=5
)

# Hybrid Search (add query_text and query_type)
results = index.similarity_search(
    query_vector=query_vector,
    query_text=query,           # Added
    query_type="hybrid",        # Added
    num_results=5
)

# Hybrid + Reranking (add reranker)
results = index.similarity_search(
    query_vector=query_vector,
    query_text=query,
    query_type="hybrid",
    reranker=DatabricksReranker(columns_to_rerank=["text"]),  # Added
    num_results=5
)
```

## Further Reading

- [Databricks Vector Search Documentation](https://docs.databricks.com/en/generative-ai/vector-search.html)
- [Understanding BM25 Algorithm](https://en.wikipedia.org/wiki/Okapi_BM25)
- [Cross-Encoders for Reranking](https://www.sbert.net/examples/applications/cross-encoder/README.html)